# Phase D full generation — 500 examples per condition (1500 total)

Generates the three training datasets via OpenRouter / claude-sonnet-4-5, validates, and runs the cross-dataset coverage gate.

Cost: ~1500 calls × ~$0.015 ≈ **$22–25** on OpenRouter. Wall-time: ~20–30 min at sem(20). Cache hits from the pilot are reused (60 calls saved).

Outputs:
- `data/demos.jsonl`, `data/first_person.jsonl`, `data/sdf.jsonl`
- Stdout: gen.validate report (leaks, token counts, coverage)
- Stdout: evals.verify_coverage report (cross-dataset gate)

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q localrouter pydantic transformers
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('clr_openrouter')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    REPO = '/content/drive/MyDrive/clr-worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

In [ ]:
# Refuse-to-overwrite guard. Bail out clearly if the full files already exist.
from pathlib import Path
for c in ['demos','first_person','sdf']:
    p = Path('data') / f'{c}.jsonl'
    if p.exists():
        raise SystemExit(f'{p} exists ({p.stat().st_size} bytes). Move/delete it before re-running, or rename.')
print('clear to generate.')

In [ ]:
# Generate, sequentially across conditions. Each condition runs internally with sem(20).
# Pilot examples cache-hit so this picks up where pilot left off.
!python -m gen.generate --condition demos        --n 500
!python -m gen.generate --condition first_person --n 500
!python -m gen.generate --condition sdf          --n 500

In [ ]:
# Per-condition validation: regex leak checks + Qwen3-tokenizer trained-token counts + coverage histograms.
!python -m gen.validate

In [ ]:
# Cross-dataset coverage gate: held-out leakage check, unknown ID check, balance check.
# Hard-fails (nonzero exit) on leak / unknown-ID; soft-warns on coverage gaps and >30% imbalance.
!python -m evals.verify_coverage

In [ ]:
# Compact summary table: per-condition n, mean tokens, # facts/traits/behaviors covered.
import json
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer
import pandas as pd

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')

rows = []
for cond in ['demos','first_person','sdf']:
    p = Path('data') / f'{cond}.jsonl'
    rs = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    tokens = [len(tok(r['assistant'], add_special_tokens=False).input_ids) for r in rs]
    fact_c = Counter(); trait_c = Counter(); beh_c = Counter()
    for r in rs:
        for fid in r.get('fact_ids', []): fact_c[fid] += 1
        for tid in r.get('trait_ids', []): trait_c[tid] += 1
        for bid in r.get('behavior_ids', []): beh_c[bid] += 1
    rows.append({
        'condition': cond,
        'n': len(rs),
        'tok_min': min(tokens), 'tok_mean': round(sum(tokens)/len(tokens), 1), 'tok_max': max(tokens),
        'tok_oor': sum(1 for t in tokens if t < 100 or t > 500),
        'unique_facts': len(fact_c),
        'unique_traits': len(trait_c),
        'unique_beh': len(beh_c),
    })
df = pd.DataFrame(rows)
df

In [ ]:
# Persist the three datasets to Drive root so they survive a runtime restart.
import shutil
from pathlib import Path
if 'COLAB_RELEASE_TAG' in os.environ:
    drive_data = Path('/content/drive/MyDrive/clr-worktest-data')
    drive_data.mkdir(parents=True, exist_ok=True)
    for c in ['demos','first_person','sdf']:
        src = Path('data') / f'{c}.jsonl'
        dst = drive_data / f'{c}.jsonl'
        if src.exists():
            shutil.copy2(src, dst)
            print(f'  saved {dst} ({dst.stat().st_size} bytes)')